# Byte Pair Encoding (BPE) — Step by Step

BPE is a subword tokenization algorithm originally from data compression.
It is used in modern LLMs (GPT, RoBERTa, …) to split text into subword tokens.

**Core idea:** Start with individual characters, then repeatedly merge the most
frequent pair of adjacent tokens — until a target vocabulary size is reached.

---

### Why subword tokenization?

| Method | Problem |
|---|---|
| Word-level | Large vocabulary, unknown words (OOV) |
| Character-level | Very long sequences, no semantic units |
| **Subword (BPE)** | **Compact vocabulary, handles rare/unknown words** |

## Step 1 — Define a small corpus

We use a tiny corpus with **word frequencies** (how often each word appears).
The `</w>` marker signals the end of a word — this lets BPE distinguish
`low` inside a compound from `low` at the end of a word.

In [ ]:
# word -> frequency in the corpus
vocab = {
    ('l', 'o', 'w', '</w>')         : 5,
    ('l', 'o', 'w', 'e', 'r', '</w>'): 2,
    ('n', 'e', 'w', 'e', 's', 't', '</w>'): 6,
    ('w', 'i', 'd', 'e', 's', 't', '</w>'): 3,
}

print("Initial vocabulary (each word split into characters + </w>):")
for word, freq in vocab.items():
    print(f"  {freq}x  {' '.join(word)}")

## Step 2 — Count all adjacent token pairs

For every word in the vocabulary we slide a window of size 2 over the token sequence
and count how often each pair appears (weighted by the word's frequency).

In [ ]:
from collections import defaultdict

def get_pair_stats(vocab):
    """Count frequency of every adjacent token pair across the whole vocabulary."""
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        for i in range(len(word) - 1):
            pairs[(word[i], word[i + 1])] += freq
    return pairs

pairs = get_pair_stats(vocab)

print("Pair frequencies (sorted):")
for pair, freq in sorted(pairs.items(), key=lambda x: -x[1]):
    print(f"  {freq}x  {pair}")

## Step 3 — Find the most frequent pair

The pair with the highest count is selected for merging.

In [ ]:
best_pair = max(pairs, key=pairs.get)
print(f"Most frequent pair: {best_pair}  ({pairs[best_pair]}x)")

## Step 4 — Merge the best pair

Every occurrence of the winning pair in all words is replaced by a single new token.

In [ ]:
def merge_pair(pair, vocab):
    """Replace every occurrence of `pair` in all words with the merged token."""
    merged_token = "".join(pair)
    new_vocab = {}
    for word, freq in vocab.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new_word.append(merged_token)   # merge
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_vocab[tuple(new_word)] = freq
    return new_vocab

vocab = merge_pair(best_pair, vocab)

print(f"After merging {best_pair} -> '{''.join(best_pair)}':")
for word, freq in vocab.items():
    print(f"  {freq}x  {' '.join(word)}")

## Step 5 — Repeat the process

Steps 2–4 are repeated. Each iteration creates one new merged token and adds it
to the **merge list** (the learned rules). Below we run several merges manually
so the effect is easy to follow.

In [ ]:
# Reset to the original vocabulary for a clean demonstration
vocab = {
    ('l', 'o', 'w', '</w>')              : 5,
    ('l', 'o', 'w', 'e', 'r', '</w>')   : 2,
    ('n', 'e', 'w', 'e', 's', 't', '</w>'): 6,
    ('w', 'i', 'd', 'e', 's', 't', '</w>'): 3,
}

merges = []   # will record every merge rule in order
NUM_MERGES = 8

for step in range(1, NUM_MERGES + 1):
    pairs = get_pair_stats(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    merges.append(best)
    vocab = merge_pair(best, vocab)

    merged_token = "".join(best)
    print(f"Merge {step:2d}:  {best[0]!r:10} + {best[1]!r:12} -> {merged_token!r:15}  (freq {pairs[best]})")
    print(f"         Vocabulary after merge:")
    for word, freq in vocab.items():
        print(f"           {freq}x  {' '.join(word)}")
    print()

## Step 6 — The learned merge rules

The ordered list of merge rules **is** the BPE model.
To tokenize new text, apply the rules in the same order.

In [ ]:
print("Learned merge rules (in order):")
for i, (a, b) in enumerate(merges, 1):
    print(f"  Rule {i:2d}: {a!r:12} + {b!r:12}  ->  {(a+b)!r}")

## Step 7 — Build the token vocabulary (token → ID)

After training, we extract every unique token that appears in the final vocabulary
and assign a numeric ID to each one.

In [ ]:
all_tokens = sorted(set(token for word in vocab.keys() for token in word))
token_to_id = {token: idx for idx, token in enumerate(all_tokens)}

print("Token  ->  ID")
for token, idx in token_to_id.items():
    print(f"  {token!r:20}  {idx}")

## Step 8 — Tokenize a new word

To tokenize a word not seen during training:
1. Split it into individual characters + `</w>`
2. Apply every learned merge rule in order
3. Look up each resulting token in `token_to_id`

In [ ]:
def tokenize(word, merges):
    """Apply BPE merge rules to a single word."""
    tokens = list(word) + ["</w>"]
    print(f"  Start:  {tokens}")

    for rule in merges:
        i = 0
        new_tokens = []
        merged = False
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == rule:
                new_tokens.append("".join(rule))
                i += 2
                merged = True
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
        if merged:
            print(f"  Rule {merges.index(rule)+1:2d} ({rule[0]!r}+{rule[1]!r}):  {tokens}")

    # Strip </w> from the last token for display
    if tokens and tokens[-1] == "</w>":
        tokens = tokens[:-1]
    elif tokens and tokens[-1].endswith("</w>"):
        tokens[-1] = tokens[-1][:-4]   # remove </w> suffix

    return tokens

test_word = "lowest"
print(f"Tokenizing: '{test_word}'")
result = tokenize(test_word, merges)
print(f"\nFinal tokens: {result}")

## Step 9 — Map tokens to IDs

Each token is looked up in `token_to_id`.
`None` means the token was never seen during training (OOV).

In [ ]:
ids = [token_to_id.get(t, None) for t in result]

print(f"Token  ->  ID")
for token, idx in zip(result, ids):
    status = str(idx) if idx is not None else "None (OOV — not in vocabulary)"
    print(f"  {token!r:20}  {status}")

oov_count = ids.count(None)
print(f"\nTotal tokens: {len(ids)},  OOV: {oov_count}")

## Step 10 — Try an unknown word

BPE gracefully handles words never seen during training: it falls back to
smaller known subword pieces — or, in the worst case, individual characters.

In [ ]:
unknown_word = "newest"
print(f"Tokenizing: '{unknown_word}'")
result_unk = tokenize(unknown_word, merges)
ids_unk = [token_to_id.get(t, None) for t in result_unk]

print(f"\nFinal tokens: {result_unk}")
print(f"Token IDs:    {ids_unk}")
print(f"OOV tokens:   {ids_unk.count(None)} / {len(ids_unk)}")

---
## Summary

| Step | What happens |
|------|--------------|
| 1 | Split corpus words into characters + `</w>` |
| 2 | Count all adjacent token pairs (weighted by frequency) |
| 3 | Select the most frequent pair |
| 4 | Merge that pair into a single new token everywhere |
| 5 | Repeat steps 2–4 for `N` merges |
| 6 | The ordered merge rules = the BPE model |
| 7 | Build `token → ID` mapping from the final vocabulary |
| 8 | Tokenize new text by applying merge rules in order |
| 9 | Look up token IDs; `None` → OOV |

> **Key insight:** BPE never produces a truly unknown token for languages it was trained on —
> the worst case is a sequence of individual characters, all of which are in the vocabulary.